# Crypto Portfolio Analytics

Discovers live perp positions and spot balances in configured Hyperliquid wallets, uses Hyperliquid daily candles for perp markets and CoinGecko prices and market data for supported assets, then calculates correlations and constrained long/short full-Kelly weights and reports portfolio performance, risk, and factor estimates. Implementation lives in `crypto_portfolio_analytics_implementation.ipynb`.


## Load implementation

Set the wallet addresses to analyze, then load the companion implementation and create the analytics object.


In [ ]:
import json
from pathlib import Path

WALLET_ADDRESSES = [
    '0x...c248',
    '0x...07f2',
]


implementation_path=Path('crypto_portfolio_analytics_implementation.ipynb')
implementation=json.loads(implementation_path.read_text(encoding='utf-8'))
namespace={'__name__':'crypto_portfolio_analytics'}
for cell in implementation['cells']:
    if cell['cell_type']=='code':
        source=''.join(cell['source'])
        if source.strip():
            exec(compile(source,f'{implementation_path} cell','exec'),namespace)
CryptoPortfolioAnalytics=namespace['CryptoPortfolioAnalytics']
analysis=CryptoPortfolioAnalytics(data_dir='data', wallet_addresses=WALLET_ADDRESSES)


## Market data refresh

Discovers current nonzero perp positions and spot balances. Perp prices come from Hyperliquid daily candles (including HIP-3 markets); CoinGecko supplies spot-only prices and market-cap/volume data without requiring an API key. Assets without enough common daily returns against BTC are reported and skipped. Edit `WALLET_ADDRESSES` above to select wallets.


In [ ]:
analysis.refresh_market_data()


## Price history

Loads assets discovered from wallet positions and spot balances, plus BTC for benchmark comparisons.


In [ ]:
analysis.load_price_history()


## Daily returns and correlations

Calculates the complete daily-return sample and Pearson correlations.


In [ ]:
analysis.calculate_returns()


## Correlation heatmap

Displays the daily return correlation matrix as an interactive Plotly heatmap.


In [ ]:
analysis.plot_correlations()


## Constrained long/short full-Kelly portfolio

Optimizes signed asset weights from historical returns; the remaining net balance is held as cash. Long/short signs are chosen by the optimizer and are not forced to match current wallet position sides.


In [ ]:
analysis.optimize_portfolio()


## Portfolio performance and risk metrics

Displays the Plotly portfolio-vs-BTC chart and performance and risk tables. Return and volatility annualization use 365 calendar days. Arithmetic annualized return is the daily mean multiplied by 365, not a forecast or compounded return. Historical VaR is an empirical quantile; Normal VaR is a Gaussian-model comparison. The observation and tail counts show how limited the 99% historical estimate is.


In [ ]:
analysis.analyze_performance()


## CoinGecko factor model

Fits factor exposures and displays the annual factor estimates and model fit.


In [ ]:
analysis.fit_factor_model()


## Save analysis outputs

Writes calculated tables and series to CSV files in `data/`.


In [ ]:
analysis.save_outputs()
